# You need to run the source_file notebook first later this notebook

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
# In BRONZE_TABLE lets add a new customer record and update some customers records
bronze_data = [
    (1, 'Sai Aditya', 999.99, 2026),  # existing id=1, updated amount
    (6, 'Maya', 712.50, 2026)          # new row
]

# Add timestamp HERE before merge
updates_df = spark.createDataFrame(
    bronze_data,
    ['id', 'name', 'amount', 'year']
).withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(F.current_timestamp(), "America/Chicago"),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

# MERGE into Delta table
delta_table = DeltaTable.forName(spark, "bronze_table")

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "name"        : "source.name",
    "amount"      : "source.amount",
    "year"        : "source.year",
    "ingested_at" : "source.ingested_at"   # <-- timestamp refreshes on update
}).whenNotMatchedInsert(values={
    "id"          : "source.id",
    "name"        : "source.name",
    "amount"      : "source.amount",
    "year"        : "source.year",
    "ingested_at" : "source.ingested_at"   # <-- timestamp set on insert
}).execute()

Lets Delete a Row from the Bronze Table


In [0]:
%sql
delete from bronze_table where id = 2

Now lets see what data is changed in the table and what got updated or inserted

In the following data you can see the _commit_version. What is the change_type also.


In [0]:
changes_df = spark.read.format("delta").option('readChangeData',True).option('startingVersion',1).table("bronze_table")
changes_df.display()

lets create a version table first for saving  latest commit version

In [0]:
version_data = [
    (1,)
]
version_df = spark.createDataFrame(version_data, ["last_version"])
version_df.write.format("delta").mode("overwrite").saveAsTable("versions_table")

In [0]:
# lets fetch the lastest version value from the version table
latest_max_value = spark.read.format("delta").table("versions_table").agg({"last_version": "max"}).collect()[0][0]
print(latest_max_value)


In [0]:
# lets create a temp table with tha latest changes from bronze_table using the latest_max_value of the commit version

spark.sql(f"select * from (select *, row_number()over(partition by id order by _commit_timestamp desc) as RN from table_changes('bronze_table',{latest_max_value}) WHERE _change_type in ('update_postimage','insert','delete')) where RN = 1").createOrReplaceTempView("temp_changes")

Lets do the merge operation between bronze and silver table

In [0]:
%sql
select * from silver_table

In [0]:
spark.sql("""
    MERGE INTO silver_table
    USING temp_changes
    ON silver_table.id = temp_changes.id
    WHEN MATCHED AND temp_changes._change_type= 'update_postimage' THEN
        UPDATE SET silver_table.name = temp_changes.name, silver_table.amount = temp_changes.amount, silver_table.year = temp_changes.year, silver_table.ingested_at = temp_changes.ingested_at
    WHEN MATCHED and temp_changes._change_type= 'delete' THEN
        DELETE
    WHEN NOT MATCHED and temp_changes._change_type= 'insert' THEN
        INSERT (id, name, amount,year, ingested_at)
        VALUES (temp_changes.id, temp_changes.name, temp_changes.amount, temp_changes.year, temp_changes.ingested_at)
""")

In [0]:
latest_commit_version = spark.sql("SELECT max(_commit_version) FROM table_changes ('bronze_table', 1)").collect()[0][0]
print(f"UPDATE versions_table SET last_version={latest_commit_version}")
spark.sql(f"UPDATE versions_table SET last_version={latest_commit_version}")

In [0]:
# Now lets check silver_table to see if the changes are reflected
spark.sql("select * from silver_table").display()

In [0]:
# we need to update the gold table with latest data
gold_df = spark.sql("""
    WITH aggregated AS (
        SELECT 
            name,
            year,
            SUM(amount) AS total_amount
        FROM silver_table
        GROUP BY name, year
    )
    SELECT 
        name,
        year,
        total_amount,
        DENSE_RANK() OVER (PARTITION BY year ORDER BY total_amount DESC) AS rank
    FROM aggregated
    ORDER BY year, rank ASC
""")

final_df_gold = gold_df.withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Chicago"
        ),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

final_df_gold.write.mode("overwrite").saveAsTable("gold_table")

In [0]:
%sql
select * from gold_table

In [0]:
# Lets Check how it runs by changing the data in bronze table
# In BRONZE_TABLE lets add a new customer record and update some customers records
bronze_data = [
    (1, 'Sai Aditya', 1205.26, 2026),  # existing id=1, updated amount
    (6, 'Maya', 1620.25, 2026)          # new row
]

# Add timestamp HERE before merge
updates_df = spark.createDataFrame(
    bronze_data,
    ['id', 'name', 'amount', 'year']
).withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(F.current_timestamp(), "America/Chicago"),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

# MERGE into Delta table
delta_table = DeltaTable.forName(spark, "bronze_table")

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "name"        : "source.name",
    "amount"      : "source.amount",
    "year"        : "source.year",
    "ingested_at" : "source.ingested_at"   # <-- timestamp refreshes on update
}).whenNotMatchedInsert(values={
    "id"          : "source.id",
    "name"        : "source.name",
    "amount"      : "source.amount",
    "year"        : "source.year",
    "ingested_at" : "source.ingested_at"   # <-- timestamp set on insert
}).execute()

In [0]:
# lets create a temp table with tha latest changes from bronze_table using the latest_max_value of the commit version

spark.sql(f"select * from (select *, row_number()over(partition by id order by _commit_timestamp desc) as RN from table_changes('bronze_table',{latest_max_value}) WHERE _change_type in ('update_postimage','insert','delete')) where RN = 1").createOrReplaceTempView("temp_changes")

In [0]:
spark.sql("""
    MERGE INTO silver_table
    USING temp_changes
    ON silver_table.id = temp_changes.id
    WHEN MATCHED AND temp_changes._change_type= 'update_postimage' THEN
        UPDATE SET silver_table.name = temp_changes.name, silver_table.amount = temp_changes.amount, silver_table.year = temp_changes.year, silver_table.ingested_at = temp_changes.ingested_at
    WHEN MATCHED and temp_changes._change_type= 'delete' THEN
        DELETE
    WHEN NOT MATCHED and temp_changes._change_type= 'insert' THEN
        INSERT (id, name, amount,year, ingested_at)
        VALUES (temp_changes.id, temp_changes.name, temp_changes.amount, temp_changes.year, temp_changes.ingested_at)
""")

In [0]:
latest_commit_version = spark.sql("SELECT max(_commit_version) FROM table_changes ('bronze_table', 1)").collect()[0][0]
print(f"UPDATE versions_table SET last_version={latest_commit_version}")
spark.sql(f"UPDATE versions_table SET last_version={latest_commit_version}")

In [0]:
# we need to update the gold table with latest data
gold_df = spark.sql("""
    WITH aggregated AS (
        SELECT 
            name,
            year,
            SUM(amount) AS total_amount
        FROM silver_table
        GROUP BY name, year
    )
    SELECT 
        name,
        year,
        total_amount,
        DENSE_RANK() OVER (PARTITION BY year ORDER BY total_amount DESC) AS rank
    FROM aggregated
    ORDER BY year, rank ASC
""")

final_df_gold = gold_df.withColumn(
    "ingested_at",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Chicago"
        ),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

final_df_gold.write.mode("overwrite").saveAsTable("gold_table")

In [0]:
%sql
select * from gold_table